<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2014%20-%202026-05-12%20-%20Dashboards%20con%20Streamlit/clase_14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 14 · Dashboards interactivos con Streamlit

**Fecha:** Martes 12 de mayo de 2026  
**Módulo:** Módulo 4 · Productización y entregables  

---

## 🗓️ Plan de la Semana 4

| Día | Fecha | Tema |
|-----|-------|------|
| 14 | Mar 12 may | Dashboards con Streamlit |
| 15 | Mié 13 may | APIs con FastAPI |
| 16 | Jue 14 may | Buenas prácticas y portafolio |
| **17** | **Vie 15 may** | **Presentaciones finales — ENTREGA FINAL** |

### Entregable de cierre: Semana 4 – Producto de Software Analítico

Al cierre del **Día 17 (viernes 15 de mayo)** cada equipo presenta y entrega:

- Comparación de al menos dos modelos con justificación del seleccionado y métricas finales (Precision, Recall, F1, ROC).
- Dashboard en Streamlit con visualizaciones clave, predicciones e interpretación de hallazgos.
- API en FastAPI con endpoints documentados (Swagger).
- Repositorio GitHub: código modularizado, documentación interna, `README` con instrucciones y `requirements.txt`.

## Introducción

Streamlit convierte un script de Python en una app web. Hoy construimos un dashboard que carga el dataset, muestra KPIs y un gráfico interactivo con filtros.

## 1. ¿Por qué Streamlit existe?

<img src="https://streamlit.io/images/brand/streamlit-mark-color.svg" width="120" alt="Streamlit logo">

Antes de 2019, si querías que otros usaran tu análisis tenías que: aprender Flask/Django, HTML, CSS, JavaScript, montar un backend, un frontend, autenticación, despliegue... o mandar un notebook por correo. Ninguna opción buena.

**Streamlit** (Adrien Treuille, Thiago Teixeira, Amanda Kelly, 2019) resolvió esto con una idea simple: que cualquier script Python se convierta en una app web con solo añadir unas líneas. Funciona tan bien que en 2022 Snowflake lo compró por 800 millones de dólares.

**Paradigma reactivo**: cada vez que el usuario cambia un widget, Streamlit **re-ejecuta el script entero** de arriba a abajo. Eso simplifica enormemente el modelo mental (no hay callbacks ni estado complejo) pero obliga a usar **cache** para no recalcular datos pesados cada vez.

## 2. Widgets y layout esenciales

| Widget | Función |
|---|---|
| `st.title()`, `st.header()`, `st.write()` | Texto |
| `st.metric()` | Números grandes (KPIs) |
| `st.sidebar.*` | Panel lateral (filtros) |
| `st.selectbox()`, `st.multiselect()` | Selección |
| `st.slider()`, `st.date_input()` | Rangos y fechas |
| `st.columns()` | Layout en columnas |
| `st.tabs()` | Pestañas |
| `st.plotly_chart()`, `st.pyplot()`, `st.altair_chart()` | Gráficos |
| `st.dataframe()` | Tabla interactiva |
| `st.map()` | Mapa simple con lat/lon |

El **cache** `@st.cache_data` guarda el resultado de la función; la próxima vez que se llame con los mismos argumentos, devuelve el resultado sin recalcular. Imprescindible para cargas de datos.

## 3. Arquitectura recomendada

Un dashboard limpio separa 3 responsabilidades:

1. **Capa de datos** (`data.py`): funciones que cargan y cachean.
2. **Capa de lógica** (`features.py`): cálculos y transformaciones.
3. **Capa de UI** (`dashboard.py`): solo widgets y llamadas.

Esto facilita probar y reutilizar. Si alguien necesita los mismos KPIs en un reporte mensual, puede importarlos sin copiar código.

## 4. Esqueleto del dashboard · guardar como `app/dashboard.py`

In [ ]:
DASHBOARD = """
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title='Análisis Titanic', page_icon='🚢', layout='wide')

@st.cache_data
def cargar_datos():
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    return pd.read_csv(url)

df = cargar_datos()

st.title('🚢 Análisis del Titanic')
st.caption('Dashboard demostrativo — seminario EDA')

# ---------- Filtros ----------
with st.sidebar:
    st.header('Filtros')
    clase = st.multiselect(
        'Clase',
        sorted(df['Pclass'].unique()),
        default=sorted(df['Pclass'].unique()),
    )
    sexo = st.multiselect('Sexo', df['Sex'].unique(), default=list(df['Sex'].unique()))

f = df[df['Pclass'].isin(clase) & df['Sex'].isin(sexo)]

# ---------- KPIs ----------
c1, c2, c3 = st.columns(3)
c1.metric('Pasajeros', f"{len(f):,}")
c2.metric('% Supervivencia', f"{f['Survived'].mean() * 100:.1f}%")
c3.metric('Edad media', f"{f['Age'].mean():.1f}")

st.divider()

# ---------- Gráficos ----------
tab1, tab2 = st.tabs(['Demografía', 'Supervivencia'])

with tab1:
    st.plotly_chart(
        px.histogram(f, x='Age', color='Sex', nbins=30, barmode='overlay',
                     title='Distribución de edad'),
        use_container_width=True,
    )

with tab2:
    g = f.groupby(['Pclass','Sex'])['Survived'].mean().reset_index()
    st.plotly_chart(
        px.bar(g, x='Pclass', y='Survived', color='Sex', barmode='group',
               title='% supervivencia por clase y sexo'),
        use_container_width=True,
    )

    with st.expander('Ver datos filtrados'):
        st.dataframe(f)
"""
print(DASHBOARD)

## 5. Ejecución local

```bash
pip install streamlit plotly
streamlit run app/dashboard.py
```

Abre automáticamente `http://localhost:8501`. Cada vez que guardas el archivo, Streamlit ofrece recargar. Cambios de widgets re-ejecutan el script; cambios de código requieren "Rerun".

## 6. Despliegue gratuito en Streamlit Cloud

1. Empujar el código a GitHub (público o privado).
2. Entrar a https://streamlit.io/cloud e iniciar sesión con GitHub.
3. "New app" → seleccionar repo, rama y archivo principal.
4. Configurar variables de entorno si las necesitas.
5. Listo. La app queda en `https://<tu-app>.streamlit.app`.

Alternativas para proyectos más grandes: Hugging Face Spaces, Render, Railway, o un VPS con Docker.

## 7. Patrones de diseño útiles

- **Un dashboard, un propósito**. No meter 10 reportes en la misma app.
- **KPIs arriba, gráficos abajo, detalle expandible**. Los usuarios escanean de arriba hacia abajo.
- **Filtros en la barra lateral**, contenido en el centro.
- **Usar `st.tabs`** para separar historias relacionadas.
- **No mostrar números sin contexto**: cada KPI con un delta o una referencia (`st.metric("Ventas", 1200, delta=150)`).

---

## 🧑‍🤝‍🧑 Trabajo en grupo sobre el caso

- Creen un `app/dashboard.py` con al menos 2 KPIs y 2 gráficos filtrables.
- Publiquen el dashboard en Streamlit Cloud (opcional pero recomendado).

## 📦 Entregable del día

Carpeta `app/` con el dashboard funcional (o captura si no publican).

---

## 📚 Lecturas adicionales

Para profundizar después de la clase, en [`Lecturas.md`](./Lecturas.md) encontrarás una curaduría de artículos técnicos y de negocios en español (4 por día), con copia local en la subcarpeta [`lecturas/`](./lecturas/) cuando la fuente lo permite.

### 🎬 Video recomendado

<iframe width="720" height="405" src="https://www.youtube.com/embed/bssw63IiSxc" title="Introducción a Streamlit · Crea aplicaciones de datos en Python con poco código" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

**[Introducción a Streamlit · Crea aplicaciones de datos en Python con poco código](https://www.youtube.com/watch?v=bssw63IiSxc)** · El Loco de los Datos

_De la instalación a una app funcional, con la arquitectura reactiva de Streamlit._